In [23]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import random
import time
%matplotlib inline

In [24]:
# Read in all the words
words = open('names.txt', 'r').read().splitlines()
print("Words: ", words[:10])
print("Length: ", len(words))

Words:  ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']
Length:  32033


In [25]:
# Build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = { ch:i+1 for i,ch in enumerate(chars) }
stoi['.'] = 0
itos = { i:ch for ch, i in stoi.items() }

vocab_size = len(itos)
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [26]:
# Build the dataset
block_size = 3
X, Y = [], []

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xtest, Ytest = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [27]:
# MLP revisited
n_embd = 10
n_hidden = 64

# initialize weights
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
# Layer1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/np.sqrt(n_embd * block_size)
b1 = torch.randn((n_hidden), generator=g) * 0.01
# Layer2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0.01

# BatchNorm parameters
bngain = torch.randn((1, n_hidden))* 0.1 + 1.0
bnbias = torch.randn((1, n_hidden))* 0.1

# Note: Initializing many params in non-standward ways
# because sometimes initializing with e.g. all zeros could mask an incorrec
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2,bngain, bnbias]

print("Total parameters: ", sum(p.numel() for p in parameters))

for p in parameters:
    p.requires_grad = True

Total parameters:  4137


In [28]:
batch_size = 32
n = batch_size # a short variable for convenience

# minibatch construct
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

In [29]:
# Forward pass,"chunked" intosmaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into a dense vector
embcat = emb.view(emb.shape[0], -1) # flatten the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani # difference from mean
bndiff2 = bndiff**2 # square it
bnvar = 1/(n-1)*(bndiff2.sum(0, keepdim=True)) # Note: Bessel's correction. Division by (n-1) instead of n

epsilon = 1e-5
bnvar_inv = (bnvar + epsilon)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact)
# Linear layer 2
logits = h @ W2 + b2 # output layer
# Cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(dim=1, keepdim=True).values
norm_logits  = logits - logit_maxes # subtract max for numerical stability

counts = norm_logits.exp()
counts_sum= counts.sum(dim=1, keepdim=True)
counts_sum_inv = counts_sum**-1 # using 1.0/counts_sum doesn't result in backprop to be exact
probs = counts * counts_sum_inv
logprobs= probs.log()
loss = -logprobs[torch.arange(n), Yb].mean()

# PyTorch Backward pass
for p in parameters:
    p.grad = None

tensor_dict = {
    "logprobs": logprobs,
    "probs": probs,
    "counts": counts,
    "counts_sum": counts_sum,
    "counts_sum_inv": counts_sum_inv,
    "norm_logits": norm_logits,
    "logit_maxes": logit_maxes,
    "logits": logits,
    "h": h,
    "hpreact": hpreact,
    "bnraw": bnraw,
    "bnvar": bnvar,
    "bnvar_inv": bnvar_inv,
    "bndiff": bndiff,
    "bndiff2": bndiff2,
    "hprebn": hprebn,
    "bnmeani": bnmeani,
    "embcat": embcat,
    "emb": emb
}

for t in tensor_dict.keys():
    tensor_dict[t].retain_grad()
    print(f"{t} shape -> {tensor_dict[t].shape}")

loss.backward()
loss

logprobs shape -> torch.Size([32, 27])
probs shape -> torch.Size([32, 27])
counts shape -> torch.Size([32, 27])
counts_sum shape -> torch.Size([32, 1])
counts_sum_inv shape -> torch.Size([32, 1])
norm_logits shape -> torch.Size([32, 27])
logit_maxes shape -> torch.Size([32, 1])
logits shape -> torch.Size([32, 27])
h shape -> torch.Size([32, 64])
hpreact shape -> torch.Size([32, 64])
bnraw shape -> torch.Size([32, 64])
bnvar shape -> torch.Size([1, 64])
bnvar_inv shape -> torch.Size([1, 64])
bndiff shape -> torch.Size([32, 64])
bndiff2 shape -> torch.Size([32, 64])
hprebn shape -> torch.Size([32, 64])
bnmeani shape -> torch.Size([1, 64])
embcat shape -> torch.Size([32, 30])
emb shape -> torch.Size([32, 3, 10])


tensor(3.2865, grad_fn=<NegBackward0>)

In [30]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

### Exercise 1: Backprop through the whole thing manually
Backpropagating through all of the variables as they are defined in the forward pass above, one by one <br />
<b>References</b>: 
- https://robotchinwag.com/posts/the-tensor-calculus-you-need-for-deep-learning/
- https://explained.ai/matrix-calculus/
- https://math.oxford.emory.edu/site/math117/besselCorrection/

In [35]:
# d_loss/d_logprobs
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[torch.arange(n), Yb] = -1.0 / n

# d_loss/d_probs
dprobs = (1.0/probs) * dlogprobs

# d_loss/d_counts_sum_inv
# Note: Handle broadcasting
'''
c = a * b, but with tensors: a[3X3] * b[3X1]
a11 * b1, a12 * b1, a13 * b1
a21 * b2, a22 * b2, a23 * b2
a31 * b3, a32 * b3, a33 * b3
'''
dcounts_sum_inv = (counts * dprobs).sum(dim=1, keepdim=True)

# d_loss/d_counts - first branch
dcounts = counts_sum_inv * dprobs

# d_loss/d_counts_sum
dcounts_sum = (-counts_sum**(-2)) * dcounts_sum_inv

# d_loss/d_counts - second branch
dcounts += torch.ones_like(counts) * dcounts_sum

# d_loss/d_norm_logits
dnorm_logits = counts * dcounts

# d_loss/d_logits - first branch
dlogits = dnorm_logits.clone() # both have same shape (32, 27)

# d_loss/d_logit_maxes
# Note: logit_maxes.shape = (32, 1) so have to handle broadcasting rule
dlogit_maxes = -(dnorm_logits).sum(1, keepdim=True)

# d__loss/d_logits - second branch
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

# d_loss/d_h
dh = dlogits @ W2.T

# d_loss/d_W2
dW2 = h.T @ dlogits

# d_loss/d_b2
db2 = dlogits.sum(dim=0)

# d_loss/d_hpreact
dhpreact = (1.0 - h**2) * dh

# d_loss/d_bngain
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)

# d_loss/d_bnraw
dbnraw = bngain * dhpreact

# d_loss/d_bnbias
dbnbias = 1.0*dhpreact.sum(0, keepdim=True)

# d_loss/d_bndiff
dbndiff = bnvar_inv * dbnraw

# d_loss/d_bnvar_inv
dbnvarinv = (bndiff * dbnraw).sum(0, keepdim=True)

# d_loss/d_bnvar
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvarinv

# d_loss/d_bndiff2
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar

# d_loss/d_bndiff
dbndiff += 2 * bndiff * dbndiff2

# d_loss/d_hprebn - first branch
dhprebn = dbndiff.clone()

# d_loss/d_bnmeani
# dbnmeani = (-torch.ones_like(bndiff) * dbndiff).sum(0)
dbnmeani = (-dbndiff).sum(0)

# d_loss/d_hprebn - second branch
dhprebn += 1.0/n*(torch.ones_like(hprebn) * dbnmeani)

# d_loss/dembcat
dembcat = dhprebn @ W1.T

# d_loss/dW1
dW1 = embcat.T @ dhprebn

# d_loss/db1
db1 = dhprebn.sum(0)

# d_loss/demb
demb = dembcat.view(emb.shape)

# d_loss/dC
# Better watch video to understand this!
dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        idx = Xb[i][j]
        dC[idx] += demb[i, j]

# compare with Pytorch
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('dbnvar_inv', dbnvarinv, bnvar_inv)
cmp('dbnvar', dbnvar, bnvar)
cmp('dbndiff2', dbndiff2, bndiff2)
cmp('dbndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

### Exercise 2: Backprop through cross_entropy but in one go
Look at the mathematical expression of the loss, take the derivative, simplify the expression and write it out

Now:
```python
loss_fast = F.cross_entropy(logits, Yb)
```

# Calculation of specific gradients considering Python broadcasting

### Calculating $\frac{\partial L}{\partial counts_{sum\_inv}}$
Let X be (32, n) and Y be (32, 1). When you compute:

`Z = X * Y`

Python broadcasts Y across all n columns of X. So Z is (32, n) where every column of X gets multiplied elementwise by Y.

Say there's some scalar loss L downstream. You're given $\frac{\partial L}{\partial Z}$ (shape 32, n) and you want:
- $\frac{\partial L}{\partial X}$ — same shape as X → (32, n)
- $\frac{\partial L}{\partial Y}$ — same shape as Y → (32, 1)

---

**Intuition first**

**Derivative w.r.t. X**

Each element $X_{ij}$ only affects $Z_{ij}$ — and $Z_{ij} = X_{ij} \times Y_{i}$. So the derivative of L w.r.t. $X_{ij}$ is just $Y_i$ scaled by the incoming gradient at that position.

Nothing tricky here — no broadcasting effect on this side.

**Derivative w.r.t. Y**

This is where broadcasting makes it interesting. Each $Y_i$ was used **n times** — once for every column in row i. So $Y_i$ affects $Z_{i,1}, Z_{i,2}, ... Z_{i,n}$ — all n elements in row i of Z.

By chain rule, you need to **sum up all the gradient contributions** coming back from each of those n positions. Broadcasting forward = summing backward. This is the key insight.

---

**The Math**

**Setting up notation**

Let:
- $Z = X \odot \hat{Y}$ where $\hat{Y}$ is Y broadcast to shape (32, n)
- $Z_{ij} = X_{ij} \cdot Y_{i}$ for $i \in [1..32], j \in [1..n]$
- $L$ is a scalar loss, $\frac{\partial L}{\partial Z}$ is given, shape (32, n)

---

**Derivative w.r.t. X**

$$Z_{ij} = X_{ij} \cdot Y_i$$

By chain rule:

$$\frac{\partial L}{\partial X_{ij}} = \frac{\partial L}{\partial Z_{ij}} \cdot \frac{\partial Z_{ij}}{\partial X_{ij}}$$

Since $\frac{\partial Z_{ij}}{\partial X_{ij}} = Y_i$:

$$\frac{\partial L}{\partial X_{ij}} = \frac{\partial L}{\partial Z_{ij}} \cdot Y_i$$

In matrix form:

$$\boxed{\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Z} \odot \hat{Y}}$$

where $\hat{Y}$ is Y broadcast to (32, n). Same shape as X. Clean and simple.

---

**Derivative w.r.t. Y**

Each $Y_i$ appears in **all n columns** of row i in Z:

$$Z_{i,1} = X_{i,1} \cdot Y_i, \quad Z_{i,2} = X_{i,2} \cdot Y_i, \quad \ldots \quad Z_{i,n} = X_{i,n} \cdot Y_i$$

By chain rule, summing over all paths through which $Y_i$ affects L:

$$\frac{\partial L}{\partial Y_i} = \sum_{j=1}^{n} \frac{\partial L}{\partial Z_{ij}} \cdot \frac{\partial Z_{ij}}{\partial Y_i}$$

Since $\frac{\partial Z_{ij}}{\partial Y_i} = X_{ij}$:

$$\frac{\partial L}{\partial Y_i} = \sum_{j=1}^{n} \frac{\partial L}{\partial Z_{ij}} \cdot X_{ij}$$

In matrix form — this is a row-wise dot product, which is a sum along columns (axis=1):

$$\boxed{\frac{\partial L}{\partial Y} = \text{sum}\left(\frac{\partial L}{\partial Z} \odot X, \text{ axis=1, keepdims=True}\right)}$$

Result shape is (32, 1) — matches Y. ✓

---

**The General Rule for Broadcasting**

This generalizes cleanly:

> **Broadcasting forward = summing gradients backward along the broadcasted dimensions**

Whenever a variable was broadcast across a dimension during the forward pass, during backprop you sum the incoming gradients along that same dimension to get back to the original shape.

In PyTorch, when you call `.backward()`, it automatically handles this. It tracks which dimensions were broadcast and reduces gradients accordingly.

### Calculation of $\frac{\partial L}{\partial counts}$

X is (32, n). You compute:

`X_sum = X.sum(1, keepdims=True)`

This sums across axis=1 (columns), giving X_sum of shape (32, 1). Each row of X gets collapsed into a single number.

So:

$$X\_sum_i = \sum_{j=1}^{n} X_{ij} \quad \text{for } i \in [1..32]$$

You're given $\frac{\partial L}{\partial X\_sum}$ of shape (32, 1) and want $\frac{\partial L}{\partial X}$ of shape (32, n).

---

**Intuition first**

Think about what summing does — it takes n values and collapses them into 1.

So $X\_sum_i$ depends on **all n elements** in row i equally. Every element $X_{ij}$ contributed equally to $X\_sum_i$ — none was weighted more than another.

Now flip to the backward pass. The gradient coming back from $X\_sum_i$ needs to be **distributed back equally** to all n elements in row i.

Summing forward = broadcasting gradient backward.

Notice this is the **exact reverse** of the previous example:
- Previous: broadcast forward → sum backward
- This: sum forward → broadcast backward

They're inverses of each other. That's the elegant symmetry here.

---
**Setting up notation**

$$X\_sum_i = \sum_{j=1}^{n} X_{ij}$$

You want $\frac{\partial L}{\partial X_{ij}}$ for all i, j.

By chain rule:

$$\frac{\partial L}{\partial X_{ij}} = \frac{\partial L}{\partial X\_sum_i} \cdot \frac{\partial X\_sum_i}{\partial X_{ij}}$$

Now compute $\frac{\partial X\_sum_i}{\partial X_{ij}}$:

$$\frac{\partial X\_sum_i}{\partial X_{ij}} = \frac{\partial}{\partial X_{ij}} \sum_{k=1}^{n} X_{ik} = 1$$

Because in the sum $\sum_k X_{ik}$, the derivative w.r.t. any single $X_{ij}$ is just 1 — all other terms vanish.

Therefore:

$$\frac{\partial L}{\partial X_{ij}} = \frac{\partial L}{\partial X\_sum_i} \cdot 1 = \frac{\partial L}{\partial X\_sum_i}$$

This means every element in row i gets the **same gradient** — the scalar gradient from $X\_sum_i$ copied n times across that row.

In matrix form — broadcast $\frac{\partial L}{\partial X\_sum}$ from (32, 1) back to (32, n):

$$\boxed{\frac{\partial L}{\partial X} = \frac{\partial L}{\partial X\_sum} \cdot \mathbf{1}^T}$$

where $\mathbf{1}^T$ is a row vector of ones of length n. This is just broadcasting (32, 1) → (32, n).

---
**The Symmetry — putting both examples together**

| Operation | Forward | Backward |
|---|---|---|
| `X * Y` (Y broadcast) | Y copied across n columns | Sum gradients across n columns |
| `X.sum(axis=1)` | n values collapsed to 1 | 1 value copied to n columns |

These are exact inverses. Broadcasting and summing are **transpose operations** of each other in the gradient sense. This is a general principle — whatever you do in the forward pass, the backward pass does the "transpose" of that operation.

This is actually why PyTorch's autograd works so elegantly — every operation just needs to know its own local gradient rule, and the chain rule handles the rest.

### Calculation of $\frac{\partial L}{\partial logits}$ - second branch

**Intuition first**

Remember with sum:
- Every element in the row contributed equally to the output
- So gradient was distributed equally back to all n elements — broadcast 1 back to n

With max it's completely different:
- Only **one element** in each row "won" — the maximum value
- All other elements had **zero influence** on the output
- So gradient flows back only to the winner — everyone else gets zero

This is called a **sparse gradient** — most elements get zero, only the max element gets the incoming gradient.

---
**The Math**

$$Y_i = \max_{j} X_{ij} \quad \text{for } i \in [1..m]$$

Y has shape (m, 1). You're given $\frac{\partial L}{\partial Y}$ of shape (m, 1) and want $\frac{\partial L}{\partial X}$ of shape (m, n).

**Derivative of max**

For a single row i, let $j^*$ be the index of the maximum element:

$$j^* = \arg\max_{j} X_{ij}$$

The derivative of $Y_i$ w.r.t. each element in row i:

$$\frac{\partial Y_i}{\partial X_{ij}} = \begin{cases} 1 & \text{if } j = j^* \\ 0 & \text{otherwise} \end{cases}$$

This can be written cleanly using an indicator function:

$$\frac{\partial Y_i}{\partial X_{ij}} = \mathbb{1}[j = \arg\max_k X_{ik}]$$

**Applying chain rule**

$$\frac{\partial L}{\partial X_{ij}} = \frac{\partial L}{\partial Y_i} \cdot \frac{\partial Y_i}{\partial X_{ij}} = \frac{\partial L}{\partial Y_i} \cdot \mathbb{1}[j = \arg\max_k X_{ik}]$$

In matrix form:

$$\boxed{\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \odot \text{one\_hot}(\arg\max(X, \text{axis=1}), \text{depth=n})}$$

Where one\_hot converts the argmax indices into a (m, n) binary mask — exactly one 1 per row at the position of the maximum.

---
**The tricky edge case — ties**

What if two elements in a row are equal and both happen to be the maximum?

$$X_{i,3} = X_{i,7} = \max_j X_{ij}$$

Mathematically the gradient is undefined — max is not differentiable at ties. In practice PyTorch breaks ties by giving the gradient to the **first** occurrence and zero to all others. This is a convention, not a mathematical truth.

This is one reason max is used carefully in neural networks — it's non-differentiable at ties and has sparse gradients. **Softmax** is often preferred because it's a smooth, differentiable approximation of max that distributes gradients to all elements.

### Calculation of derivatives for equation: $Y = W @ x + b$
#### where `@` is matrix multiplication
#### Calculating $\frac{\partial L}{\partial W}$, $\frac{\partial L}{\partial x}$, $\frac{\partial L}{\partial b}$

In `Y = W@x + b`:
- W is the weight matrix
- x is the input
- b is the bias
- Y is the output

You need three gradients — $\frac{\partial L}{\partial W}$, $\frac{\partial L}{\partial x}$, $\frac{\partial L}{\partial b}$.

The bias gradient is the easiest — same as the sum case we already did. The interesting ones are W and x.

**Key intuition for matrix multiply gradients**

There's a beautiful pattern here worth memorizing:

> The gradient w.r.t. one operand involves multiplying by the **transpose** of the other operand.

This isn't arbitrary — it falls out naturally from the math. Let me show why.

---

**The Math**

**Setup**

Let:
- W be (m, n)
- x be (n, 1)
- b be (m, 1)
- Y = W@x + b, shape (m, 1)
- L is scalar loss, $\frac{\partial L}{\partial Y}$ is given, shape (m, 1)

---

**Derivative w.r.t. b**

$$Y_i = \sum_j W_{ij} x_j + b_i$$

$$\frac{\partial Y_i}{\partial b_i} = 1$$

By chain rule:

$$\frac{\partial L}{\partial b_i} = \frac{\partial L}{\partial Y_i} \cdot 1 = \frac{\partial L}{\partial Y_i}$$

In matrix form:

$$\boxed{\frac{\partial L}{\partial b} = \frac{\partial L}{\partial Y}}$$

Same shape (m, 1). Gradient just passes straight through. ✓

---

**Derivative w.r.t. x**

$$Y_i = \sum_k W_{ik} x_k + b_i$$

Each $x_j$ contributes to **every** output $Y_i$ through the weight $W_{ij}$. So by chain rule, sum over all paths through which $x_j$ influences L:

$$\frac{\partial L}{\partial x_j} = \sum_i \frac{\partial L}{\partial Y_i} \cdot \frac{\partial Y_i}{\partial x_j}$$

Since $\frac{\partial Y_i}{\partial x_j} = W_{ij}$:

$$\frac{\partial L}{\partial x_j} = \sum_i \frac{\partial L}{\partial Y_i} \cdot W_{ij}$$

Notice this is summing over the **i-th row** of W for each j — that's exactly what $W^T$ does when multiplied by a vector:

$$\boxed{\frac{\partial L}{\partial x} = W^T \frac{\partial L}{\partial Y}}$$

Shape check: $W^T$ is (n, m), $\frac{\partial L}{\partial Y}$ is (m, 1) → result is (n, 1) = shape of x. ✓

---

**Derivative w.r.t. W**

$$Y_i = \sum_k W_{ik} x_k + b_i$$

Each weight $W_{ij}$ only affects $Y_i$ directly:

$$\frac{\partial Y_i}{\partial W_{ij}} = x_j$$

By chain rule:

$$\frac{\partial L}{\partial W_{ij}} = \frac{\partial L}{\partial Y_i} \cdot \frac{\partial Y_i}{\partial W_{ij}} = \frac{\partial L}{\partial Y_i} \cdot x_j$$

This is an outer product — for each i,j combination you're multiplying the i-th gradient by the j-th input:

$$\boxed{\frac{\partial L}{\partial W} = \frac{\partial L}{\partial Y} \cdot x^T}$$

Shape check: $\frac{\partial L}{\partial Y}$ is (m, 1), $x^T$ is (1, n) → result is (m, n) = shape of W. ✓

---

**The Transpose Pattern**

Notice the beautiful symmetry:

$$\frac{\partial L}{\partial x} = W^T \frac{\partial L}{\partial Y}$$

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial Y} \cdot x^T$$

In both cases the gradient w.r.t. one operand involves the **transpose of the other operand** multiplied by the incoming gradient. This is the pattern worth memorizing.

Intuitively — transpose "reverses" the matrix multiplication, routing gradients back to where they came from.

---

**Why this matters so much for deep learning**

Every linear layer in a neural network is exactly `W@x + b`. Backprop through a deep network is just applying these three gradient formulas repeatedly through each layer — chain rule connecting them together.

When you see papers talk about "gradient flow" through deep networks, it's literally this transpose pattern applied layer after layer. If W has very large or very small values, $W^T$ amplifies or shrinks gradients as they flow backward — which is exactly why weight initialization and batch normalization matter so much, connecting back to the Karpathy videos you've been watching.

### Calculation of derivative for $Y = X[z]$ where $X$, $Y$, $z$ are tensors

`X[z]` is just fancy indexing, you're using z as a lookup table of indices into X.

Think of X as a dictionary with 27 entries (like 27 characters in a vocabulary), each entry being a vector of size 10. z contains (32, 3) indices — you're looking up 32×3 = 96 entries from X, possibly with repetition.

The result Y is (32, 3, 10) — each index in z got replaced by its corresponding row in X.

Now for gradients — the key question is:

*"Which rows of X were used, and how many times?"*

Because a row of X might be looked up **multiple times** if the same index appears multiple times in z. Each time it was used, it receives a gradient contribution. All contributions must be summed back — exactly like the broadcasting case.

---

**The Math**

**Setup**

- X is (27, 10)
- z is (32, 3) — integer indices, values in range [0..26]
- Y = X[z], shape (32, 3, 10)
- L is scalar loss, $\frac{\partial L}{\partial Y}$ is given, shape (32, 3, 10)

Note: z contains **integers** — no gradient flows through z itself. Only X gets a gradient.

**Derivative w.r.t. X**

Each element $Y_{i,j,k}$ comes directly from $X[z_{i,j}, k]$:

$$Y_{i,j,k} = X_{z_{ij}, k}$$

So:

$$\frac{\partial Y_{i,j,k}}{\partial X_{r,k}} = \mathbb{1}[r = z_{ij}]$$

By chain rule, the gradient flowing back to row r of X is the sum of all incoming gradients from positions in Y that looked up row r:

$$\frac{\partial L}{\partial X_{r,k}} = \sum_{i,j} \frac{\partial L}{\partial Y_{i,j,k}} \cdot \mathbb{1}[r = z_{ij}]$$

In plain English — for each row r of X, find all positions (i,j) in z where $z_{ij} = r$, and sum up the corresponding gradients from $\frac{\partial L}{\partial Y}$:

$$\boxed{\frac{\partial L}{\partial X} = \text{scatter\_add}\left(\frac{\partial L}{\partial Y}, \text{index}=z\right)}$$

Shape check: result is (27, 10) = shape of X. ✓

---

**The scatter\_add pattern**

PyTorch implements this internally as `scatter_add` — it scatters the incoming gradients back to their source rows in X and adds them up when the same row was used multiple times.

This is the exact reverse of the gather operation (indexing forward):

| Direction | Operation | Name |
|---|---|---|
| Forward | `X[z]` — pick rows by index | Gather |
| Backward | Sum gradients back to source rows | Scatter Add |

Gather forward → Scatter Add backward. Another clean inverse pair, just like sum/broadcast.

---

**Why this matters — Embedding layers**

This operation IS the embedding layer in transformers and language models. X is the embedding matrix (vocabulary\_size, embedding\_dim), z is your token indices, and Y is the sequence of embeddings.

During backprop, gradients flow back through this exact scatter\_add operation — updating only the rows of the embedding matrix that were actually used in this batch. Rows corresponding to rare words get updated rarely. Rows for common words get large gradients because they're looked up constantly.

This is why embedding layers are sometimes trained with special optimizers — the gradient is very sparse (most rows get zero gradient in any given batch) which makes standard gradient descent inefficient.